In [3]:

import json
import hashlib
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import sys
sys.path.append("..")
from utils.balanced_builders_hdf import *
from utils.losses import *
from utils.recon_error import *
from utils.trainer import *
from models.autoencoder_classifier import *
from config import *

In [5]:
sg = build_balanced_sg_loaders_from_h5(
    h5_path=xrd_dataset,
    min_count_sg=20000,
    per_class_cap_sg=20000,
    batch_size=256,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=4,
)

print(sg["num_classes"])
print(sg["sizes"])

22
{'train': 352000, 'val': 44000, 'test': 44000}


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepConvAutoencoderClassifier(
    input_length=sg["input_len"],
    latent_dim=64,         
    cls_dim=128,            
    num_classes=sg["num_classes"],
    use_projection_head=True
).to(device)

def make_optimizer(model, lr=1e-3, wd=1e-4):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=wd
    )

optimizer = make_optimizer(model)

# ===== Balanced: good reconstruction + stronger classifier, mild contrastive =====
params = {
    # Reconstruction vs classification balance
    "recon_mul":   1.5,    # strong reconstruction
    "alpha_cls":   7.0,    # strong classifier signal
    "beta_smooth": 0.02,   # smoothness regularization

    # Augmentation / contrastive
    "noise_std":   0.02,   # denoising effect
    "lambda_ntx":  0.03,   # NT-Xent (weak)
    "lambda_sup":  0.2,    # supervised contrastive
}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Disable augmentation/contrastive completely
params["noise_std"]  = 0.0
params["lambda_ntx"] = 0.0
params["lambda_sup"] = 0.0

num_epochs = 300
best_val_acc = 0.0

save_path_clf   = SG_Cls
save_path_recon = SG_rec

for epoch in range(1, num_epochs + 1):
    logs = {"loss": 0.0, "recon": 0.0, "clf": 0.0, "smooth": 0.0, "ntx": 0.0, "supcon": 0.0}
    steps = 0

    # -------------------------
    # Training
    # -------------------------
    for x, y in sg["train_loader"]:
        out = train_step(
            model=model,
            optimizer=optimizer,
            x=x,
            y=y,
            epoch=epoch,
            params=params,
            device=device,
            use_augmentation=False,   
        )

        for k in logs:
            logs[k] += out[k]
        steps += 1

    for k in logs:
        logs[k] /= max(1, steps)

    # -------------------------
    # Validation
    # -------------------------
    val = val_step(model, sg["val_loader"], device)
    val_acc   = val["val_acc"]
    val_recon = val["val_recon"]

    # -------------------------
    # Save best classifier
    # -------------------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "num_classes": sg["num_classes"],
        "label_map": sg["label_map"],
        "input_len": sg["input_len"],
        "epoch": epoch,
        "best_val_acc": best_val_acc,
    }, save_path_clf)
        torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "num_classes": sg["num_classes"],
    "label_map": sg["label_map"],
    "input_len": sg["input_len"],
    "epoch": num_epochs,
    "best_val_acc": best_val_acc,
}, save_path_recon)


        print(f"✓ Saved BEST CLASSIFIER at epoch {epoch} | ValAcc = {best_val_acc:.4f}")

    # -------------------------
    # Logging
    # -------------------------
    print(
        f"Epoch {epoch:03d} | "
        f"Loss {logs['loss']:.4f} | Recon {logs['recon']:.4f} | "
        f"Clf {logs['clf']:.4f} | Smooth {logs['smooth']:.4f} | "
        f"NTX {logs['ntx']:.4f} | SupCon {logs['supcon']:.4f} || "
        f"ValAcc {val_acc:.4f} | ValRecon {val_recon:.4f}"
    )


# Save final model after full training
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "num_classes": sg["num_classes"],
    "label_map": sg["label_map"],
    "input_len": sg["input_len"],
    "epoch": num_epochs,
    "best_val_acc": best_val_acc,
}, save_path_recon)

print(f"✓ Saved FINAL model after full {num_epochs} epochs → {save_path_recon}")

✓ Saved BEST CLASSIFIER at epoch 1 | ValAcc = 0.7321
Epoch 001 | Loss 8.6424 | Recon 0.0114 | Clf 1.2322 | Smooth 0.0036 | NTX 0.0000 | SupCon 0.0000 || ValAcc 0.7321 | ValRecon 0.0077
✓ Saved BEST CLASSIFIER at epoch 2 | ValAcc = 0.8922
Epoch 002 | Loss 3.0416 | Recon 0.0069 | Clf 0.4330 | Smooth 0.0034 | NTX 0.0000 | SupCon 0.0000 || ValAcc 0.8922 | ValRecon 0.0065
✓ Saved BEST CLASSIFIER at epoch 3 | ValAcc = 0.9221
Epoch 003 | Loss 1.4248 | Recon 0.0061 | Clf 0.2022 | Smooth 0.0034 | NTX 0.0000 | SupCon 0.0000 || ValAcc 0.9221 | ValRecon 0.0060
✓ Saved BEST CLASSIFIER at epoch 4 | ValAcc = 0.9365
Epoch 004 | Loss 0.9690 | Recon 0.0056 | Clf 0.1372 | Smooth 0.0034 | NTX 0.0000 | SupCon 0.0000 || ValAcc 0.9365 | ValRecon 0.0054
Epoch 005 | Loss 0.7631 | Recon 0.0052 | Clf 0.1079 | Smooth 0.0035 | NTX 0.0000 | SupCon 0.0000 || ValAcc 0.9284 | ValRecon 0.0060
Epoch 006 | Loss 0.6405 | Recon 0.0049 | Clf 0.0904 | Smooth 0.0035 | NTX 0.0000 | SupCon 0.0000 || ValAcc 0.9293 | ValRecon 0.0